In [59]:
import torch
from torch import nn
from torchvision import datasets
from torch.utils.data import DataLoader, random_split
from torchvision.transforms import ToTensor, transforms
import torchvision.models as models
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import numpy as np

In [41]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [38]:
!unzip -q "/content/drive/MyDrive/animal_data.zip" -d "/content/dataset"

replace /content/dataset/__MACOSX/._animal_data? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/dataset/__MACOSX/animal_data/._Cat? [y]es, [n]o, [A]ll, [N]one, [r]ename: r
new name: 

In [42]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]) # Standard ImageNet values
])
full_dataset = datasets.ImageFolder(root = "/content/dataset/animal_data", transform = transform)

In [43]:
train_size = int(0.8*(len(full_dataset)))
test_size = len(full_dataset) - train_size

In [44]:
train_dataset , test_dataset = random_split(full_dataset,[train_size,test_size])

In [45]:
train_loader = DataLoader(dataset = train_dataset, shuffle = True, num_workers = 2,batch_size = 32)
test_loader = DataLoader(dataset = test_dataset, shuffle = True, num_workers = 2, batch_size = 32)

In [46]:
device = torch.device('cuda')
device

device(type='cuda')

In [52]:
model = models.resnet18(weights="DEFAULT")

for param in model.parameters():
    param.requires_grad = False
model.fc = nn.Linear(512, 15)
model.to(device)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [54]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(),lr = 1e-3)

In [55]:
def train(dataloader, model, loss_fn,optimizer):
  size = len(dataloader.dataset)
  loss = 0
  correct = 0
  model.train()
  for X,y in dataloader:
    X = X.to(device)
    y = y.to(device)

    optimizer.zero_grad()

    pred = model(X)
    loss = loss_fn(pred,y)

    loss.backward()

    optimizer.step()

    correct += (pred.argmax(1) == y).type(torch.float).sum().item()

  correct /= size
  print(f"Train Error : \n Accuracy  : {(100*correct):>0.1f} %")


In [56]:
def test(dataloader, model, loss_fn):
  size = len(dataloader.dataset)
  loss, correct = 0,0
  model.eval()
  with torch.no_grad():
    for X,y in dataloader:
      X = X.to(device)
      y = y.to(device)

      pred = model(X)

      loss = loss_fn(pred,y)

      correct += (pred.argmax(1) == y).type(torch.float).sum().item()

  correct /= size
  print(f"Test Error : \n Accuracy :{(100*correct):>0.1f}")

In [57]:
epochs  = 10
for i in range(epochs):
  print(f"Epoch {i+1}:------")
  train(train_loader,model,loss_fn,optimizer)
  test(test_loader,model,loss_fn)

print("Done")


Epoch 1:------
Train Error : 
 Accuracy  : 58.3 %
Test Error : 
 Accuracy :85.6
Epoch 2:------
Train Error : 
 Accuracy  : 88.0 %
Test Error : 
 Accuracy :93.1
Epoch 3:------
Train Error : 
 Accuracy  : 92.3 %
Test Error : 
 Accuracy :94.3
Epoch 4:------
Train Error : 
 Accuracy  : 94.6 %
Test Error : 
 Accuracy :93.6
Epoch 5:------
Train Error : 
 Accuracy  : 94.1 %
Test Error : 
 Accuracy :96.1
Epoch 6:------
Train Error : 
 Accuracy  : 95.7 %
Test Error : 
 Accuracy :95.4
Epoch 7:------
Train Error : 
 Accuracy  : 96.9 %
Test Error : 
 Accuracy :95.6
Epoch 8:------
Train Error : 
 Accuracy  : 96.7 %
Test Error : 
 Accuracy :96.1
Epoch 9:------
Train Error : 
 Accuracy  : 98.0 %
Test Error : 
 Accuracy :96.4
Epoch 10:------
Train Error : 
 Accuracy  : 98.1 %
Test Error : 
 Accuracy :95.4
Done


In [63]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            outputs = model(images)
            preds = outputs.argmax(1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

print("--- FINAL CLASSIFICATION REPORT ---")
report = classification_report(all_labels, all_preds)
print(report)

--- FINAL CLASSIFICATION REPORT ---
              precision    recall  f1-score   support

        Bear     1.0000    0.8333    0.9091        30
        Bird     0.9565    0.8800    0.9167        25
         Cat     0.9643    1.0000    0.9818        27
         Cow     0.9200    0.9583    0.9388        24
        Deer     0.8929    0.9615    0.9259        26
         Dog     1.0000    0.9167    0.9565        24
     Dolphin     1.0000    0.9643    0.9818        28
    Elephant     0.9667    0.9355    0.9508        31
     Giraffe     0.8824    1.0000    0.9375        15
       Horse     0.9677    0.9677    0.9677        31
    Kangaroo     0.9286    1.0000    0.9630        26
        Lion     0.8696    0.9524    0.9091        21
       Panda     0.9394    1.0000    0.9688        31
       Tiger     1.0000    1.0000    1.0000        26
       Zebra     1.0000    0.9583    0.9787        24

    accuracy                         0.9537       389
   macro avg     0.9525    0.9552    0.9524 